In [ ]:
import json
import pickle
from pathlib import Path
from collections import defaultdict
import random
import subprocess
from tqdm.auto import tqdm
import os

print("--- Step 0: Setup ---")

# --- Path Configuration ---
PKL_DIRS = {
    'multichain': Path("/home/qiuyk/data/multiChain_dataset_20250917/multiChain_processed_pkls"),
    'ppq': Path("/home/qiuyk/data/PPQ/PPQ_processed_pkls")
}
# Work directory. All intermediate and final files will be saved here.
WORK_DIR = Path("/home/qiuyk/data/split_workflow")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# --- Define the output file name ---
FASTA_FILE = WORK_DIR / "all_sequences.fasta"
SOURCE_MAP_FILE = WORK_DIR / "seq_id_to_source_map.json"
CLUSTER_FILE = WORK_DIR / "cluster_results_cluster.tsv"
FINAL_SPLIT_JSON = WORK_DIR / "data_splits.json"

SEED = 42
random.seed(SEED)

print("Setup complete. Working directory is ready.")
print(f"All outputs will be saved in: {WORK_DIR.resolve()}")

In [ ]:
print("--- Step 1: Extracting all sequences from .pkl files... ---")

total_chains = 0
seq_id_to_source_map = {} 

with open(FASTA_FILE, 'w') as f_out:
    for dataset_name, pkl_dir in PKL_DIRS.items():
        if not pkl_dir.exists():
            print(f"Warning: Directory not found, skipping: {pkl_dir}")
            continue
        
        pkl_files = list(pkl_dir.glob("*.pkl"))
        print(f"Found {len(pkl_files)} .pkl files in {dataset_name} dataset.")
        
        for pkl_file in tqdm(pkl_files, desc=f"Processing {dataset_name}"):
            complex_id = pkl_file.stem
            try:
                with open(pkl_file, 'rb') as f_in:
                    data = pickle.load(f_in)
                
                for chain_data in data.get('chain_features', []):
                    chain_id = chain_data['chain_id']
                    sequence = chain_data['seq']
                    
                    # The unique sequence ID format: dataset_name-ComplexID-ChainID
                    unique_seq_id = f"{dataset_name}-{complex_id}-{chain_id}"
                    
                    # Write to FASTA file
                    f_out.write(f">{unique_seq_id}\n")
                    f_out.write(f"{sequence}\n")
                    
                    # Record the source information of this sequence
                    seq_id_to_source_map[unique_seq_id] = {
                        "dataset": dataset_name,
                        "complex_id": complex_id
                    }
                    total_chains += 1
            except Exception as e:
                print(f"Error processing file {pkl_file.name}: {e}")

# Save the mapping file
with open(SOURCE_MAP_FILE, 'w') as f:
    json.dump(seq_id_to_source_map, f)

print(f"\n✅ Extraction complete! Extracted {total_chains} chain sequences.")
print(f"FASTA file saved to: {FASTA_FILE}")
print(f"Source map saved to: {SOURCE_MAP_FILE}")

In [ ]:
print("--- Step 2: Running MMseqs2 for global sequence clustering... ---")

# MMseqs2 Command Parameters
# --min-seq-id 0.3: 30% sequence consistency threshold
# -c 0.8: 80% Coverage threshold
# --threads 1: Ensure that the clustering results can be reproduced.

# mmseqs easy-cluster all_sequences.fasta cluster_results tmp --min-seq-id 0.3 -c 0.8 --cov-mode 1 --threads 1


print("✅ MMseqs2 clustering complete!")
print(f"Cluster results saved in: {WORK_DIR}")
# Check if the output file exists
assert CLUSTER_FILE.exists(), f"Error: Expected cluster file not found at {CLUSTER_FILE}"

In [ ]:
print("--- Step 3: Generating final data splits based on clusters ---")

def integrate_samples(split_reps, source_filter=None):
    
    chain_to_split_map = {member_chain: split_name for split_name, reps in split_reps.items() for r in reps for member_chain in clusters[r]}
    complex_to_chains_map = defaultdict(list)
    
    # Establish a mapping from complex_id to all its chain_ids
    for chain_id in chain_to_rep:
        complex_id = seq_id_to_source[chain_id]['complex_id']
        complex_to_chains_map[complex_id].append(chain_id)

    final_splits = {'train': set(), 'validation': set(), 'test': set()}
    
    # Obtain all unique complex_ids
    all_complex_ids = set(c['complex_id'] for c in seq_id_to_source.values())

    for complex_id in all_complex_ids:
        # If a source filter is set, but the current sample source does not match, then skip.
        dataset_source = complex_id.split('-')[0] if '-' in complex_id else 'ppq' # Assuming the prefix-less version is "PPQ"
        is_ppq_source = 'ppq' in complex_id or dataset_source == 'ppq' 
        
        # Find the dataset corresponding to the complex_id from seq_id_to_source
        # We assume that a complex_id belongs to only one dataset
        source_dataset = None
        if complex_id in complex_to_chains_map:
             # Determine the source of the dataset by using the first chain of the complex
             first_chain = complex_to_chains_map[complex_id][0]
             source_dataset = seq_id_to_source[first_chain]['dataset']

        if source_filter and source_dataset != source_filter:
            continue

        member_chains = complex_to_chains_map.get(complex_id, [])
        if not member_chains: continue

        chain_splits = {chain_to_split_map.get(c) for c in member_chains if c in chain_to_split_map}
        
        if not chain_splits: continue # If none of the chains of this complex are included in the current classification, then skip it.

        if 'test' in chain_splits:
            final_splits['test'].add(complex_id)
        elif 'validation' in chain_splits:
            final_splits['validation'].add(complex_id)
        else:
            final_splits['train'].add(complex_id)
            
    return {k: sorted(list(v)) for k, v in final_splits.items()}

# 1. Load the required files
print("Step 3.1: Loading cluster results and source map...")
with open(SOURCE_MAP_FILE, 'r') as f:
    seq_id_to_source = json.load(f)
clusters = defaultdict(list)
chain_to_rep = {}
with open(CLUSTER_FILE, 'r') as f:
    for line in f:
        representative, member = line.strip().split('\t')
        clusters[representative].append(member)
        chain_to_rep[member] = representative

# 2. Classify clusters by content
print("Step 3.2: Classifying clusters...")
ppq_related_reps, multichain_only_reps = set(), set()
for rep, members in clusters.items():
    is_ppq_related = any(seq_id_to_source[m]['dataset'] == 'ppq' for m in members)
    if is_ppq_related:
        ppq_related_reps.add(rep)
    else:
        multichain_only_reps.add(rep)
ppq_related_reps, multichain_only_reps = list(ppq_related_reps), list(multichain_only_reps)

# 3. Cluster Division Representation
print("Step 3.3: Splitting cluster representatives...")
random.shuffle(ppq_related_reps)
ft_ratios = (0.9, 0.05, 0.05)
num_ft_clusters = len(ppq_related_reps)
num_ft_train = int(num_ft_clusters * ft_ratios[0])
num_ft_val = int(num_ft_clusters * ft_ratios[1])
ft_splits_reps = {
    'train': ppq_related_reps[:num_ft_train],
    'validation': ppq_related_reps[num_ft_train : num_ft_train + num_ft_val],
    'test': ppq_related_reps[num_ft_train + num_ft_val:]
}

random.shuffle(multichain_only_reps)
pt_ratios = (0.8, 0.1, 0.1)
num_pt_clusters = len(multichain_only_reps)
num_pt_train = int(num_pt_clusters * pt_ratios[0])
num_pt_val = int(num_pt_clusters * pt_ratios[1])
pt_splits_reps = {
    'train': multichain_only_reps[:num_pt_train],
    'validation': multichain_only_reps[num_pt_train : num_pt_train + num_pt_val],
    'test': multichain_only_reps[num_pt_train + num_pt_val:]
}

# 4. Generate initial division
print("Step 3.4: Integrating samples with strict source filtering...")
# When generating the fine-tuning set, it is mandatory to select only the samples whose source is 'PPQ'
ft_splits = integrate_samples(ft_splits_reps, source_filter='ppq') 
# When generating the pre-training dataset, it is mandatory to select only the samples whose source is 'multichain'.
pt_splits = integrate_samples(pt_splits_reps, source_filter='multichain')

# 5. Final Cleanup: PDB ID Blacklist
print("Step 3.5: Final cleanup with PDB ID blacklist...")
# The blacklist is now generated based on the complex_id of all 'ppq' sources, with greater strictness.
all_ppq_complex_ids = {info['complex_id'] for info in seq_id_to_source.values() if info['dataset'] == 'ppq'}
ft_pdb_blacklist = {cid.split('_')[0] for cid in all_ppq_complex_ids}
print(f"Created a PDB ID blacklist with {len(ft_pdb_blacklist)} unique IDs from the entire PPQ dataset.")

final_pt_splits = {'train': [], 'validation': [], 'test': []}
for split_name in ['train', 'validation', 'test']:
    for complex_id in pt_splits[split_name]:
        pdb_id = complex_id.split('_')[0]
        if pdb_id not in ft_pdb_blacklist:
            final_pt_splits[split_name].append(complex_id)

# Final Output
final_splits_output = {"fine-tuning": ft_splits, "pre-training": final_pt_splits}
with open(FINAL_SPLIT_JSON, 'w') as f:
    json.dump(final_splits_output, f, indent=4)
    
print("\n✅ Split generation complete!")
print(f"Fine-tuning splits: Train({len(final_splits_output['fine-tuning']['train'])}), Val({len(final_splits_output['fine-tuning']['validation'])}), Test({len(final_splits_output['fine-tuning']['test'])})")
print(f"Pre-training splits (after cleanup): Train({len(final_splits_output['pre-training']['train'])}), Val({len(final_splits_output['pre-training']['validation'])}), Test({len(final_splits_output['pre-training']['test'])})")
print(f"Final splits file saved to: {FINAL_SPLIT_JSON}")

In [ ]:
print("--- Step 4: Verifying the splits for data leakage... ---")

with open(FINAL_SPLIT_JSON, 'r') as f:
    splits = json.load(f)

# Obtain all PDB IDs from the pre-trained validation and test sets
pt_val_pdbs = {cid.split('_')[0] for cid in splits['pre-training']['validation']}
pt_test_pdbs = {cid.split('_')[0] for cid in splits['pre-training']['test']}
pretrain_eval_pdbs = pt_val_pdbs.union(pt_test_pdbs)

# Obtain all PDB IDs of the fine-tuning set
ft_train_pdbs = {cid.split('_')[0] for cid in splits['fine-tuning']['train']}
ft_val_pdbs = {cid.split('_')[0] for cid in splits['fine-tuning']['validation']}
ft_test_pdbs = {cid.split('_')[0] for cid in splits['fine-tuning']['test']}
finetune_all_pdbs = ft_train_pdbs.union(ft_val_pdbs).union(ft_test_pdbs)

# Core Check: Whether the pre-trained evaluation set PDB and the fine-tuned set PDB have any intersection
leakage = pretrain_eval_pdbs.intersection(finetune_all_pdbs)

if not leakage:
    print("✅ Verification successful! No PDB ID leakage found between pre-training evaluation sets and the fine-tuning set.")
else:
    print(f"❌ Verification failed! Found {len(leakage)} leaking PDB IDs:")
    print(leakage)

assert not leakage, "Data leakage detected!"

In [ ]:
print("--- Step 5: Creating Training Cluster Map for Dynamic Sampling ---")

OUTPUT_MAP_FILE = WORK_DIR / "train_cluster_map.json"

# 1. Load the required files
print("Loading necessary files...")
with open(FINAL_SPLIT_JSON, 'r') as f:
    splits = json.load(f)

with open(SOURCE_MAP_FILE, 'r') as f:
    seq_id_to_source = json.load(f)

# Create two mappings from the cluster.tsv file:
# 1. Cluster represents -> Cluster member list
# 2. Cluster member -> Cluster represent
clusters = defaultdict(list)
chain_to_rep = {}
with open(CLUSTER_FILE, 'r') as f:
    for line in f:
        representative, member = line.strip().split('\t')
        clusters[representative].append(member)
        chain_to_rep[member] = representative

# 2. Core Logic: Identify all the training set samples and organize them by clusters
# Merge the training set IDs from the pre-training and fine-tuning processes into a single large training set
train_complex_ids = set(splits['pre-training']['train']) | set(splits['fine-tuning']['train'])
print(f"Found a total of {len(train_complex_ids)} unique complex IDs across all training sets.")

# Build a mapping from cluster representatives to a list of composite members belonging to the training set
train_cluster_map = defaultdict(list)

# Traverse all sequences and their sources
for seq_id, source_info in tqdm(seq_id_to_source.items(), desc="Mapping training samples to clusters"):
    complex_id = source_info['complex_id']
    
    # If this complex is included in our merged training set
    if complex_id in train_complex_ids:
        # Find the representative of the cluster to which this sequence belongs
        rep = chain_to_rep.get(seq_id)
        if rep:
            # Add the complex ID to the corresponding cluster member list
            train_cluster_map[rep].append(complex_id)

# 3. Remove duplicates and sort the member lists of each cluster
for rep, members in train_cluster_map.items():
    train_cluster_map[rep] = sorted(list(set(members)))

# 4. Save the file
with open(OUTPUT_MAP_FILE, 'w') as f:
    json.dump(train_cluster_map, f, indent=4)
    
print(f"\n✅ Successfully created training cluster map.")
print(f"Map contains {len(train_cluster_map)} training clusters.")
print(f"File saved to: {OUTPUT_MAP_FILE}")

In [ ]:
import json
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
import sys

try:
    WORK_DIR = Path("/home/qiuyk/data/split_workflow")
    OUT_DIR = Path("/home/qiuyk/data/split_workflow/for_test")
    if not WORK_DIR.exists():
        print(f"Error: The working directory {WORK_DIR} does not exist.")
        sys.exit(1)

    INPUT_SPLITS = WORK_DIR / "data_splits.json"
    INPUT_SOURCE_MAP = WORK_DIR / "seq_id_to_source_map.json"
    INPUT_CLUSTERS = WORK_DIR / "cluster_results_cluster.tsv"

    # Reversed mapping (for analysis and debugging)
    OUTPUT_SAMPLE_TO_CLUSTER_MAP = OUT_DIR / "ppq_test_sample_to_clusters.json"
    # The final optimized and loadable "data_splits" file
    OUTPUT_OPTIMIZED_SPLITS = OUT_DIR / "data_splits_optimized_test.json"
    # Only includes the list of sample IDs of the optimized test set (for ease of use)
    OUTPUT_OPTIMIZED_LIST = OUT_DIR / "ppq_refold_test_list.json"

    for f in [INPUT_SPLITS, INPUT_SOURCE_MAP, INPUT_CLUSTERS]:
        if not f.exists():
            print(f"Error: The required input file was not found: {f}.")
            sys.exit(1)

    print(f"WORK DIR: {WORK_DIR.resolve()}")
    print(f"Input Splits: {INPUT_SPLITS.name}")
    print(f"Input Source Map: {INPUT_SOURCE_MAP.name}")
    print(f"Input Clusters: {INPUT_CLUSTERS.name}")


    print("\n--- Phase 1: Establishing the 'sample -> clustering' reverse mapping ---")

    # 1. Load the PPQ test set
    print(f"Loading {INPUT_SPLITS.name}...")
    with open(INPUT_SPLITS, 'r') as f:
        original_splits_data = json.load(f)
    ppq_test_complex_ids = set(original_splits_data['fine-tuning']['test'])
    print(f"Found {len(ppq_test_complex_ids)} original PPQ test set samples.")

    # 2. Mapping of loading sequence to clustering (chain -> rep)
    print(f"Loading {INPUT_CLUSTERS.name}...")
    chain_to_rep = {}
    with open(INPUT_CLUSTERS, 'r') as f:
        for line in tqdm(f, desc="Read the clustering file..."):
            representative, member = line.strip().split('\t')
            chain_to_rep[member] = representative
    print(f"A total of {len(chain_to_rep)} mappings from chains to clusters were constructed.")

    # 3. Load the mapping from the source to the sequence (chain -> complex_id).
    print(f"加载 {INPUT_SOURCE_MAP.name}...")
    with open(INPUT_SOURCE_MAP, 'r') as f:
        seq_id_to_source = json.load(f)

    # 4. Core: Establishing Inverse Mapping (complex_id -> set(clusters)).
    sample_to_clusters_map = defaultdict(set)
    all_test_clusters = set()
    print("The test set samples are currently being mapped to the clusters they belong to...")
    for seq_id, source_info in tqdm(seq_id_to_source.items(), desc="Mapping samples"):
        complex_id = source_info['complex_id']

        # If this complex is included in our PPQ testing set
        if complex_id in ppq_test_complex_ids:
            # Find the cluster to which it belongs
            rep = chain_to_rep.get(seq_id)
            if rep:
                sample_to_clusters_map[complex_id].add(rep)
                all_test_clusters.add(rep)

    print(f"Mapping completed. The original test set covers a total of {len(all_test_clusters)} independent clusters.")

    # 5. Save this useful reverse mapping
    # (Converting the set to a list in order to enable JSON serialization)
    serializable_map = {sample: sorted(list(clusters)) for sample, clusters in sample_to_clusters_map.items()}
    with open(OUTPUT_SAMPLE_TO_CLUSTER_MAP, 'w') as f:
        json.dump(serializable_map, f, indent=4)
    print(f"✅ Saved reverse mapping file: {OUTPUT_SAMPLE_TO_CLUSTER_MAP.name}")


    print("\n--- Phase 2: Execute the greedy algorithm to select the optimal sample set ---")

    # Convert back to set to perform set operations
    remaining_samples_map = {sample: set(clusters) for sample, clusters in serializable_map.items()}
    clusters_to_cover = all_test_clusters.copy()
    optimized_test_set = []

    pbar = tqdm(total=len(clusters_to_cover), desc="In coverage clustering...")
    while clusters_to_cover and remaining_samples_map:
        best_sample = None
        best_gain = 0 # Number of newly covered clusters

        # 1. Find the samples that can cover the largest number of *uncovered* clusters
        for sample_id, clusters in remaining_samples_map.items():
            gain = len(clusters.intersection(clusters_to_cover))
            if gain > best_gain:
                best_gain = gain
                best_sample = sample_id

        # 2. If no samples can be found to cover the new cluster, then exit.
        if best_sample is None:
            break

        # 3. Select this "optimal sample"
        optimized_test_set.append(best_sample)

        # 4. Update the "Clusters to Be Covered" list
        covered_by_best = remaining_samples_map.pop(best_sample) # Remove from the candidate pool
        newly_covered_count = len(clusters_to_cover.intersection(covered_by_best))
        clusters_to_cover.difference_update(covered_by_best)

        pbar.update(newly_covered_count)

    pbar.close()

    if clusters_to_cover:
        print(f"Warning: After the algorithm finishes, there are still {len(clusters_to_cover)} clusters that have not been covered.")
        print(f"This might indicate that some clusters do not have corresponding PPQ test set samples (which should not occur theoretically).")

    print("\n--- The algorithm has been completed ---")
    print(f"Original test set sample size: {len(ppq_test_complex_ids)}")
    print(f"Optimized test set sample size: {len(optimized_test_set)}")
    print(f"Total number of clustering groups: {len(all_test_clusters)}")


    print("\n--- Phase Three: Save the final output file ---")

    # 1. Save the new "data_splits" file
    # We duplicate the original data and only replace the "fine-tuning/test" section.
    new_splits_data = original_splits_data.copy()
    new_splits_data['fine-tuning']['test'] = sorted(optimized_test_set)

    with open(OUTPUT_OPTIMIZED_SPLITS, 'w') as f:
        json.dump(new_splits_data, f, indent=4)
    print(f"✅ The optimized Data Splits file has been saved: {OUTPUT_OPTIMIZED_SPLITS.name}")
    print(f"   (This file contains all the original data, with only the 'fine-tuning' -> 'test' list replaced.)")

    # 2. Maintain a pure list of samples
    with open(OUTPUT_OPTIMIZED_LIST, 'w') as f:
        json.dump(sorted(optimized_test_set), f, indent=4)
    print(f"✅ The list of optimized sample IDs has been saved: {OUTPUT_OPTIMIZED_LIST.name}")

    print("\n--- The script has been executed ---")

except Exception as e:
    print(f"\n--- A serious error has occurred ---")
    print(f"Error type: {type(e).__name__}")
    print(f"Error message: {e}")
    sys.exit(1)